<a href="https://colab.research.google.com/github/irishsun51/AIFFEL_quest_eng/blob/master/LLM_Application/LLM01/Day1_RAG_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

last modified date : 2026.03.15  
제작 : 박광석 (모두의연구소)

# 랭체인으로 RAG 시작하기

해당 노트는 Langchain으로 RAG를 구현하기 위해 필요한
각 컴포넌트인 Document Loaders, Text splitters, Text embeddings, Vectorstores, Retriever를 다룹니다  




### Step 0 : 설치와 준비  
Langchain 설치 및 Gemini API 키를 등록하도록 합니다.  

In [27]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [37]:
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain-community langchain-core
!pip install -U langchain-text-splitters


In [28]:
import os


In [29]:
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_KEY')

In [6]:
#! curl ipinfo.io

In [30]:
from langchain_openai import ChatOpenAI

# OpenAI API를 사용하는 설정으로 변경
# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
)

In [31]:
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires aiofiles<25.0,>=22.0, but you have aiofiles 25.1.0 which is incompatible.
gradio 5.50.0 requires pydantic<=2.12.3,>=2.0, but you have pydantic 2.13.4 which is incompatible.


### Step 1 : Document Loaders 사용해보기  

Document Loader는 다양한 형태의 원본 데이터를  
LLM이 이해할 수 있는 Document 객체(text + metadata) 로 변환하는 역할을 합니다.

PDF, 웹페이지, CSV와 같이 형식이 서로 다른 문서들을 일관된 구조로 파싱하여, 이후 Chunking·Embedding·검색(Retrieval) 단계에서
바로 사용할 수 있도록 만들어줍니다.

즉, Document Loader는
**RAG 파이프라인의 가장 첫 단계에서 “데이터를 읽을 수 있는 형태로 정리하는 역할을 담당**합니다.

공식 문서에서는 지원되는 다양한 Loader 목록을 확인할 수 있습니다.
https://python.langchain.com/docs/modules/data_connection/document_loaders/

#### PDFLoader 사용  
이번 실습에서는 가장 많이 사용되는 문서 형식인 PDF 파일을 대상으로
PyPDFLoader를 사용해 문서를 불러옵니다.

실습을 위해, 질의응답에 활용하고 싶은 PDF 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

PDFLoader는 각 페이지를 하나의 Document 단위로 변환하며,
이 단계에서 생성된 문서들은 이후 Text Splitter를 통해 의미 단위로 다시 분할됩니다.

In [32]:
!pip install "pydantic<=2.12.3"

  Using cached pydantic-2.12.3-py3-none-any.whl.metadata (87 kB)
  Using cached pydantic_core-2.41.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.3 kB)
Using cached pydantic-2.12.3-py3-none-any.whl (462 kB)
Using cached pydantic_core-2.41.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (2.1 MB)
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.46.4
    Uninstalling pydantic_core-2.46.4:
      Successfully uninstalled pydantic_core-2.46.4
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.13.4
    Uninstalling pydantic-2.13.4:
      Successfully uninstalled pydantic-2.13.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unstructured-client 0.44.0 requires pydantic>=2.12.5, but you have pydantic 2.12.3 which is incompatible.
gradio 5.50.0 requires aiofiles<25.0,>

In [6]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/drive/MyDrive/Colab Notebooks/RAG/DAY1/Demian.pdf")
pages = loader.load_and_split()

/tmp/ipykernel_33532/3566485363.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [2]:
pages[0]

Document(metadata={'producer': 'Adobe Acrobat Standard DC 19 Paper Capture Plug-in', 'creator': 'ScanFix(TM) Enhanced', 'creationdate': '2015-09-10T01:40:29+00:00', 'moddate': '2019-01-30T17:47:47+01:00', 'source': '/content/drive/MyDrive/Colab Notebooks/RAG/DAY1/Demian.pdf', 'total_pages': 182, 'page': 0, 'page_label': '1'}, page_content='DEMIAN \n• \nDownloaded from https://www.holybooks.com')

In [3]:
print(pages[10])

page_content='TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut F

출력 결과를 보기 쉽게 확인하기 위해,
Document 객체 전체가 아닌 실제 텍스트 본문이 담긴 page_content만 선택하여 확인해보겠습니다.

In [4]:
print(pages[10].page_content)

TWO WOR.LDS 
Finally, out of sheer nervousness, I began to talk. I 
invented a long story of robbery, in which I featured as 
the hero. One night in the comer by the mill a friend 
and I ha.d stolen a whole sackful of apples, not just 
ordinary apples but pippins, golden pippins of the best 
kind at that. I was taking refuge in my story from the 
dangers of the moment and found no difficulty in invent­
ing and relating it. In order not to dry up too soon and 
perhaps become involved in something worse, I gave full 
rein to my narrative powers. One of us, I reported, had 
always stood guard while the other sat in the tree and 
chucked the apples down, and the sack had got so heavy 
that in the end we had to open it and leave half behind, 
but we came back half an hour later and fetched them 
too. 
I hoped for some applause at the end of my story; I 
had warmed up to the narrative aJ: last, carried away by 
my own eloquence. The two smaller boys were silent, 
waiting, Lut Franz Kromer ga

#### CSVLoader

SV 파일은 행(row) 단위로 구조화된 데이터를 담고 있는 형식으로,
LangChain의 CSVLoader를 사용하면 각 행을 하나의 Document 객체로 변환할 수 있습니다.

이렇게 변환된 문서들은 이후 PDF나 웹 문서와 동일하게
Embedding, VectorStore, Retrieval 단계에서 함께 활용할 수 있습니다.

실습을 위해, CSV 파일을 먼저 Colab 환경(또는 Drive)에 업로드해주세요.

In [8]:
from langchain_community.document_loaders import CSVLoader

loader = CSVLoader("/content/drive/MyDrive/Colab Notebooks/RAG/DAY1/titanic.csv")

data = loader.load()

In [6]:
data[:3]

[Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/RAG/DAY1/titanic.csv', 'row': 0}, page_content='PassengerId: 1\nSurvived: 0\nPclass: 3\nName: Braund, Mr. Owen Harris\nSex: male\nAge: 22\nSibSp: 1\nParch: 0\nTicket: A/5 21171\nFare: 7.25\nCabin: \nEmbarked: S'),
 Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/RAG/DAY1/titanic.csv', 'row': 1}, page_content='PassengerId: 2\nSurvived: 1\nPclass: 1\nName: Cumings, Mrs. John Bradley (Florence Briggs Thayer)\nSex: female\nAge: 38\nSibSp: 1\nParch: 0\nTicket: PC 17599\nFare: 71.2833\nCabin: C85\nEmbarked: C'),
 Document(metadata={'source': '/content/drive/MyDrive/Colab Notebooks/RAG/DAY1/titanic.csv', 'row': 2}, page_content='PassengerId: 3\nSurvived: 1\nPclass: 3\nName: Heikkinen, Miss. Laina\nSex: female\nAge: 26\nSibSp: 0\nParch: 0\nTicket: STON/O2. 3101282\nFare: 7.925\nCabin: \nEmbarked: S')]

#### 웹베이스로더  
웹베이스 로더는 웹페이지에 포함된 텍스트 콘텐츠를 직접 파싱하여 Document 객체로 변환하는 역할을 합니다.  
이를 통해 뉴스 기사, 블로그 글, 공지사항과 같은 실시간으로 업데이트되는 웹 문서를 RAG 시스템의 지식 소스로 활용할 수 있습니다.  
이번 실습에서는 실제 뉴스 기사를 예제로 사용하여,
웹페이지의 내용을 불러오고 텍스트 형태로 변환하는 과정을 살펴봅니다.  

실습에 사용할 웹페이지는 다음과 같습니다.  
https://it.chosun.com/news/articleView.html?idxno=2023092111831

In [9]:
from langchain_community.document_loaders import WebBaseLoader

In [8]:
loader = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loader.load()

#print(documents[0].page_content)

주석을 해제하고 코드를 실행하면,
해당 웹페이지에 포함된 본문 텍스트 전체를 불러와 확인할 수 있습니다.  

웹페이지, PDF, CSV 등 서로 다른 형식의 문서들이
모두 텍스트 형태로 정상적으로 파싱된 것을 확인할 수 있습니다.  

이제 이 텍스트를 **전처리(불필요한 요소 제거, 정제)** 한 뒤,
Chunking과 Embedding 단계에 활용할 수 있습니다.  

### Step2 : TextSplitters 사용해보기  
Text Splitter는 긴 텍스트 문서를 **의미를 유지한 작은 단위(Chunk)** 로 분할하는 역할을 합니다.  
LLM은 한 번에 처리할 수 있는 토큰 수에 제한이 있기 때문에, 문서를 그대로 입력하는 대신 Splitter를 통해 분할된 여러 Chunk를 입력받아 처리하게 됩니다.  

이 과정을 통해 긴 문서에서도 토큰 길이 제약을 극복하고, 필요한 부분만 효율적으로 검색할 수 있습니다.  

분할된 각 Chunk는 이후 단계에서 1:1로 Embedding되어 VectorStore에 저장되며,
이 Chunk 단위가 RAG 시스템에서 검색과 응답의 기본 단위가 됩니다.  

In [10]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

CharacterTextSplitter는
하나의 고정된 구분자(separator)를 기준으로 텍스트를 분할하는 방식입니다.
구현이 단순하고 직관적이지만,
문서 구조에 따라 분할된 Chunk가 토큰 제한을 초과하는 경우가 발생할 수 있습니다.

반면, RecursiveCharacterTextSplitter는
줄바꿈, 문장 구분자, 구두점 등 여러 구분자를 순차적으로 적용하며
텍스트를 재귀적으로 분할합니다.

이 방식은 토큰 제한을 안정적으로 만족시키는 데 유리하지만,
분할 과정에서 의미적으로 완전하지 않은 문장 단위로 잘릴 수 있다는 단점이 있습니다.  

단순한 구조의 문서나,
문단 구성이 명확한 텍스트의 경우에는 CharacterTextSplitter로도 충분합니다.

하지만 실제 서비스 환경에서는
문서 길이와 구조가 제각각인 경우가 많기 때문에,
대부분의 RAG 시스템에서는 RecursiveCharacterTextSplitter를 기본 선택지로 사용합니다.

이는 Chunk 크기를 안정적으로 제어하면서도
검색 실패를 줄이는 데 유리하기 때문입니다.

In [11]:
with open("/content/drive/MyDrive/Colab Notebooks/RAG/DAY1/state_of_the_union.txt") as f:
    text = f.read()

In [12]:
#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수입니다.
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정하는 것입니다.
text_splitter = CharacterTextSplitter(separator="\n\n", chunk_size=1000, chunk_overlap=100, length_function = len,)
chunks = text_splitter.split_text(text)

Chunk의 내용을 확인해보겠습니다

In [13]:
print(chunks[0])

Madam Speaker, Madam Vice President, our First Lady and Second Gentleman. Members of Congress and the Cabinet. Justices of the Supreme Court. My fellow Americans.  

Last year COVID-19 kept us apart. This year we are finally together again. 

Tonight, we meet as Democrats Republicans and Independents. But most importantly as Americans. 

With a duty to one another to the American people to the Constitution. 

And with an unwavering resolve that freedom will always triumph over tyranny. 

Six days ago, Russia’s Vladimir Putin sought to shake the foundations of the free world thinking he could make it bend to his menacing ways. But he badly miscalculated. 

He thought he could roll into Ukraine and the world would roll over. Instead he met a wall of strength he never imagined. 

He met the Ukrainian people. 

From President Zelenskyy to every Ukrainian, their fearlessness, their courage, their determination, inspires the world.


각 chunk의 길이를 확인해보겠습니다,

In [13]:
length = []
for chunk in chunks:
    length.append(len(chunk))

print(length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]


### 토큰 단위로 텍스트 분할해보기  
  
LLM은 문장을 단어가 아닌 토큰(token) 단위로 처리합니다.
따라서 사람이 인식하는 단어 길이나 문자 수는
실제 모델이 처리하는 입력 길이와 정확히 일치하지 않을 수 있습니다.

이로 인해 문자 수나 단어 수를 기준으로 텍스트를 분할할 경우,
모델의 입력 토큰 제한을 초과하거나
예상보다 훨씬 짧은 문맥만 전달되는 문제가 발생할 수 있습니다.

실제 서비스 환경에서는 이러한 문제를 방지하기 위해,
토큰 단위를 기준으로 텍스트를 분할하는 방식을 사용합니다.
이제 토큰 기준으로 텍스트를 분할해보겠습니다.

In [15]:
!pip install tiktoken

In [14]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

In [15]:
tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


글자 수와 토큰 수의 차이를 확인할 수 있습니다 !


### Step3 : TextEmbedding 사용해보기  
Embedding은 텍스트를 컴퓨터가 계산할 수 있는 수치 벡터(vector) 형태로 변환하는 과정입니다.
이 벡터는 문장의 표면적인 형태가 아니라, 의미적 유사성을 반영하도록 설계되어 있습니다.

변환된 벡터는
VectorStore에 저장되거나,
새로운 질의(Query) 벡터와의 유사도 계산을 통해
의미적으로 가까운 문서를 검색하는 데 사용됩니다.

이러한 변환은 대규모 말뭉치로 사전 학습된
Embedding 전용 모델을 통해 이루어지며,
RAG 시스템에서 Retrieval 성능을 결정하는 핵심 요소입니다.

이번 실습에서는
OpenAI 임베딩 모델을 사용해
텍스트를 벡터로 변환해보겠습니다.

In [18]:
import os
import openai
from google.colab import userdata

# 코랩 보안 비밀에 저장한 키를 가져와 환경 변수로 등록합니다. name :OPENAI_API_KEY 고정이어야 함.
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')

print(len(os.environ["OPENAI_API_KEY"]))


164


genai 라이브러리의 list_models 함수를 사용하여 사용 가능한 모델들의 목록을 가져옵니다.

In [21]:
# 이제 키를 따로 안 넣어줘도 자동으로 환경 변수에서 읽어옵니다.
client = openai.OpenAI()

In [22]:
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


text-embedding-3-small은 가성비가 좋고, text-embedding-3-large는 성능이 더 강력합니다.

In [23]:
from langchain_openai import OpenAIEmbeddings


embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

만약 여러분들이 Gemini를 사용하여 구축중이시라면, embedding은 지역에 따라 사용이 제한됩니다.  
주로 유럽권에서 제한되기 때문에, 다음 에러를 확인하신다면 Colab 파일의 서버 저장 위치를 확인 후, 다른 임베딩 모델로 변경해야합니다.  

Error embedding content: 400 User location is not supported for the API use.


In [ ]:
#!curl ipinfo.io

In [35]:
# 400 User location is not supported for the API use 오류가 발생한다면, 이 블록을 대신 실행해주세요

# ! pip install -q sentence_transformers

#from langchain.embeddings import HuggingFaceEmbeddings
#embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

embedding model 변수에 OpenAI 임베딩모델 혹은 huggingface의 임베딩모델이 할당되었을 것입니다.  
embed_documents 멤버 함수를 사용하여 새 문장을 변환해보겠습니다  

In [24]:
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

임베딩으로 잘 변환되었는지 확인해보겠습니다  

In [25]:
print(embeddings[1])

[0.006427764892578125, -0.020843505859375, -0.0216217041015625, 0.0235443115234375, -0.0428466796875, -0.004589080810546875, 0.039703369140625, 0.05267333984375, -0.0018796920776367188, -0.0237579345703125, 0.0138092041015625, 0.0159912109375, -0.044464111328125, -0.00011223554611206055, -0.007648468017578125, 0.02557373046875, 0.025421142578125, 0.03546142578125, -0.051788330078125, 0.01232147216796875, 0.02484130859375, -0.00047087669372558594, 0.0193939208984375, 0.07525634765625, -0.002044677734375, -0.009185791015625, 0.0168304443359375, -0.007129669189453125, 0.02655029296875, -0.04852294921875, 0.043182373046875, -0.042572021484375, 0.01163482666015625, -0.03289794921875, -0.0333251953125, -0.04620361328125, 0.0019054412841796875, 0.0225677490234375, -0.018096923828125, 0.03253173828125, 0.0293426513671875, 0.011749267578125, -0.01446533203125, 0.003063201904296875, 0.024322509765625, 0.04815673828125, -0.01354217529296875, 0.04937744140625, -0.0406494140625, 0.04400634765625, 0

In [38]:
len(embeddings[1])

1536

새로운 쿼리를 넣어, 임베딩끼리 유사도를 계산해보겠습니다

In [26]:
import numpy as np
from numpy import dot
from numpy.linalg import norm
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

In [27]:
query = ["this is red fruit"]

In [28]:
e_query = embedding_model.embed_documents(query)
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7478529746500573
0.489938276348812
0.40842404995484394


빨간 사과와 빨간 과일의 유사도가 많이 높게 나왔습니다!  
  
임베딩 모델은 사용 언어나 필요에 따라 다양하게 교체하여 사용할 수 있습니다.  
해당 링크에서 여러 목록을 확인하실 수 있습니다.  
https://python.langchain.com/docs/integrations/text_embedding/

### Step4 : VectorStore 사용해보기
VectorStore는 텍스트를 Embedding 모델을 통해 벡터(vector)로 변환한 뒤, 이를 저장하고 관리하는 저장소입니다.
이 저장소는 단순한 데이터 보관 공간이 아니라,
벡터 간의 유사도를 빠르게 계산하고 탐색하기 위한 인덱싱 구조를 함께 포함하고 있습니다.

문서나 쿼리가 Embedding된 이후에는,
VectorStore를 통해 의미적으로 유사한 벡터를 효율적으로 검색할 수 있으며,
이 과정이 RAG 시스템의 Retrieval 단계를 담당하게 됩니다.

대표적인 VectorStore로는
Chroma, FAISS 등이 있으며,
각각 로컬 환경과 대규모 서비스 환경에서 널리 사용됩니다.

이번 실습에서는
구성이 단순하고 로컬 환경에서 바로 사용할 수 있는
ChromaDB를 사용해 VectorStore를 구성해보겠습니다.

In [42]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 698.8 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 36.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 49.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 3.7 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling opentelemet

In [1]:
!pip install langchain-chroma

In [29]:
from langchain_chroma import Chroma

In [3]:
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

제일 처음에 사용했던, PDF를 다시 사용하도록 합니다!  

In [30]:
# 위에서 사용했던 코드입니다
loader = PyPDFLoader("/content/drive/MyDrive/Colab Notebooks/RAG/DAY1/Demian.pdf")
pages = loader.load_and_split()

text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)
docs = text_splitter.split_documents(pages)

In [ ]:
#!pip show chromadb

Chroma에 임베딩 시킵니다  

In [31]:
db = Chroma.from_documents(docs, embedding_model)


이제 쿼리를 날려보겠습니다

In [32]:
query = "how Demian look like?"
docs = db.similarity_search(query)

In [33]:
print(docs[0].page_content)

DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

Face, features, looks like 등 데미안의 생김새를 담고 있는 페이지가 출력되었습니다  
굉장히 빠른 속도로 검색했습니다!  

### Step5 : Retriever 사용해보기  

Retriever는 사용자의 질문을 Embedding 모델을 통해 벡터로 변환한 뒤,
VectorStore에 저장된 문서 벡터들과 비교하여
의미적으로 가장 유사한 문서(Chunk)를 찾아 반환하는 역할을 합니다.

즉, Retriever는
RAG 시스템에서 “어떤 정보를 LLM에게 참고 자료로 줄 것인가”를 결정하는 핵심 컴포넌트이며,
검색 결과의 품질이 곧 최종 답변의 품질로 이어집니다.
  

In [34]:
!pip install -U langchain langchain-classic

In [35]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA


긴 문서 전체를 한 번에 LLM에 전달하는 대신,
Retriever와 LLM을 결합한 RetrievalQA 체인을 사용하여
문서에서 질문과 관련된 부분만 검색하고,
그 결과를 바탕으로 답변을 생성합니다.

이를 통해 길이가 긴 문서에서도
토큰 제한을 넘지 않으면서, 근거 기반의 질의응답을 수행할 수 있습니다.

In [39]:
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_openai import ChatOpenAI

# OpenAI 모델로 변경
# streaming=True와 callbacks 설정을 통해 실시간 출력을 활성화합니다.
llm = ChatOpenAI(
    model="gpt-4o",              # 또는 "gpt-4o-mini"
    temperature=0.0,
    streaming=True,              # 실시간 출력을 켭니다
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

체인의 종류와 검색(Retrieval) 방식,
그리고 그에 따른 주요 파라미터를 설정합니다.

이 단계에서는
Retriever가 어떤 전략으로 문서를 검색할지,
그리고 몇 개의 문서를 LLM에게 전달할지를 결정하게 됩니다.
이 선택은 최종 답변의 품질과 직접적으로 연결됩니다.

예를 들어,
MMR(Maximal Marginal Relevance) 방식은
쿼리와의 유사도뿐만 아니라 문서 간의 중복을 줄이고 다양성을 확보하는 재정렬(Re-ranking) 전략입니다.

실무 환경에서는 단일 문서에 정보가 몰리는 것을 방지하고,
LLM이 보다 풍부한 문맥을 참고하도록 하기 위해
MMR 방식이 자주 사용됩니다.

In [44]:
qa = RetrievalQA.from_chain_type(llm, chain_type="stuff",
                                 retriever=db.as_retriever(
                                     search_type="mmr",
                                     search_kwargs={"k": 3, "fetch_k" : 10}),
                                 return_source_documents=True)

위 코드에서 짚고 넘어갈 파라미터는 다음과 같습니다  
🔹 chain_type="stuff"

검색된 문서(Chunk)를 그대로 하나의 Prompt에 모두 삽입하는 방식입니다.
구조가 단순하고 이해하기 쉬워,
RAG 구조를 처음 학습하거나 프로토타입을 만들 때 적합합니다.
단점으로는 문서 수가 많아질 경우
토큰 사용량이 빠르게 증가할 수 있습니다.
실무에서는 초기 검증 단계에서는 stuff를,
문서 수가 많아지면 map_reduce나 refine 방식으로 확장합니다.  

🔹 retriever

VectorStore에서 어떤 문서를 검색할지 결정하는 검색 모듈입니다.
검색 전략과 파라미터 설정에 따라 LLM이 참고하는 정보의 범위와 품질이 달라집니다.  

🔹 search_type="mmr"

MMR(Maximal Marginal Relevance) 검색 방식을 사용합니다. 쿼리와의 유사도뿐만 아니라, 문서 간 중복을 줄여 다양한 문맥을 확보하는 Re-ranking 전략입니다. 실무 환경에서 단일 문서 편향을 줄이기 위해 자주 사용됩니다

🔹 search_kwargs={"k": 3, "fetch_k": 10}  
- fetch_k  
VectorStore에서 우선적으로 가져올 후보 문서 개수입니다. Re-ranking 이전 단계에서 사용됩니다.
- k  
최종적으로 LLM에게 전달할 문서(Chunk)의 개수입니다.

일반적으로 fetch_k > k 로 설정하여 후보 풀을 넉넉히 확보한 뒤, 품질 좋은 문서만 선별하는 방식을 사용합니다.  

🔹 return_source_documents=True

답변 생성에 사용된 원문 문서(Chunk)를 함께 반환합니다. 이를 통해 답변의 출처를 사용자에게 표시하거나 검색 품질을 디버깅하고 RAG 성능을 평가할 수 있습니다. 실무 서비스에서는 거의 필수적으로 사용하는 옵션입니다.

In [45]:
query = "how demian looks like"
result = qa(query)

Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face has an almost feminine element and is different from the rest of the people around him. The narrator perceives Demian as being like an animal, a spirit, or an image, and describes him as unimaginably different from others.

마크다운 형식으로 출력해봅니다

In [42]:
from IPython.display import Markdown, display
display(Markdown(result["result"]))

Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His face has an almost feminine element and is different from the rest of the people around him. The narrator perceives Demian as being like an animal, a spirit, or an image, and describes him as unimaginably different from others. The narrator also notes that Demian's face is similar to a drawing he made, which he later realizes expresses his own inner self or fate.

RAG를 사용하지 않은 llm 호출도 시도해보세요!

In [43]:
llm2 = ChatOpenAI(
    model="gpt-4o")
request = llm2.invoke("how demian looks like")
display(Markdown(request.content))


If you're referring to the character Demian from Hermann Hesse's novel "Demian," then there isn't a definitive description of his physical appearance in the book. The narrative focuses more on his enigmatic and almost ethereal presence rather than specific physical traits. Demian is portrayed as a charismatic and mysterious figure with a profound impact on the protagonist, Emil Sinclair. His allure is more about his intellectual and spiritual influence rather than his looks. Readers often imagine him based on his symbolic role rather than concrete physical details.

### Quiz
결과의 어떤 부분을 관찰하였을 때, RAG 시스템의 결과를 신뢰할 수 있겠다 생각하셨나요?  

### Answer  
원문에서 답변의 출처를 확인할 수 있었습니다.

## 6. 완성 예제  
앞에서 진행한 내용으로, Demian을 다시 한번 읽어봅시다!  
완성하여 제출해주세요~


필요한 라이브러리를 모두 다운받습니다  

In [3]:
# 설치
!pip install -U -q langchain langchain-openai
!pip install -U -q langchain langchain-community langchain-core
!pip install -U langchain-text-splitters

# PDF
!pip install -q pypdf pdf2image docx2txt pdfminer unstructured #의존성 모듈을 설치합니다
!pip install tiktoken

# VectorStore
!pip install chromadb
!pip install langchain-chroma
!pip install --upgrade opentelemetry-api
!pip install --upgrade opentelemetry-sdk

# Retriever
!pip install -U langchain langchain-classic

Text splitter 사용을 위한 준비입니다

In [1]:
import os

# API KEY 정의
from google.colab import userdata
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

# LLM설정
from langchain_openai import ChatOpenAI

# 모델명은 필요에 따라 "gpt-4o", "gpt-4-turbo", "gpt-3.5-turbo" 등으로 바꿀 수 있습니다.
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.0,
)

### Step 1 Document loader

In [52]:
# PDF loader-----------------
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader("/content/drive/MyDrive/Colab Notebooks/RAG/DAY1/Demian.pdf")
pages = loader.load_and_split()

# WEB loader ----------------
from langchain_community.document_loaders import WebBaseLoader

loaderWeb = WebBaseLoader("https://it.chosun.com/news/articleView.html?idxno=2023092111831")
documents = loaderWeb.load()

### Step 2 Text splitters

In [58]:
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter

with open("/content/drive/MyDrive/Colab Notebooks/RAG/DAY1/state_of_the_union.txt") as f:
  text = f.read()

#len은 어떤 기준으로 chunk size를 잴 것인가?의 기준이 되는 함수
#chunk_overlap은 chunk의 앞뒤로 다른 chunk와 설정한 size까지 겹칠 수 있도록 설정
text_splitter = CharacterTextSplitter(separator="\n\n",chunk_size=1000, chunk_overlap=100, length_function = len,)
chunk = text_splitter.split_text(text)

In [66]:
import tiktoken
tokenizer = tiktoken.get_encoding("cl100k_base")

def tiktoken_len(text):
    tokens = tokenizer.encode(
        text
    )
    return len(tokens)

tiktoken_length = []
for chunk in chunks:
    tiktoken_length.append(tiktoken_len(chunk))

print(length)
print(tiktoken_length)

[939, 995, 810, 962, 994, 883, 957, 952, 928, 915, 993, 702, 900, 950, 957, 958, 967, 996, 796, 866, 888, 966, 964, 977, 998, 948, 925, 924, 989, 965, 938, 936, 981, 965, 771, 982, 972, 977, 984, 999, 968, 801]
[197, 198, 163, 190, 203, 182, 195, 197, 206, 205, 218, 148, 188, 205, 216, 215, 209, 224, 176, 187, 201, 197, 201, 215, 222, 202, 203, 204, 229, 206, 184, 204, 197, 194, 156, 200, 194, 221, 203, 225, 209, 187]


### Step 3 Vector Empeddings

In [59]:
import openai

# embedding 모델 종류
client = openai.OpenAI()
for model in client.models.list():
    if "embedding" in model.id:
        print(model.id)

text-embedding-ada-002
text-embedding-3-small
text-embedding-3-large


In [61]:
from langchain_openai import OpenAIEmbeddings

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")
# embedding 문장들
embeddings = embedding_model.embed_documents(
    [
        "This is red apple.",
        "This is yellow banana.",
        "This is green lime.",
    ]
)

In [62]:
import numpy as np
from numpy import dot
from numpy.linalg import norm

# cos 유사도 계산 함수
def cos_sim(A, B):
  return dot(A, B)/(norm(A)*norm(B))

# query
query = ["this is red fruit"]
e_query = embedding_model.embed_documents(query)

# embedding 문장들과 query와의 cos 유사도 계산 출력.
print(cos_sim(embeddings[0], e_query[0]))
print(cos_sim(embeddings[1], e_query[0]))
print(cos_sim(embeddings[2], e_query[0]))

0.7478929665954975
0.4898791411166917
0.4083791543048118


- PDF 문서를 읽어와서 의미 단위로 쪼갠 뒤, 벡터 데이터베이스(Chroma)에 저장하고, 사용자의 질문과 가장 유사한 본문을 찾아내는 전 RAG(검색 증강 생성)의 데이터 배치 및 검색 파이프라인

In [68]:
from langchain_chroma import Chroma

# 1. 문서 로드 (Document Loading) ---------------------------------------------
loader = PyPDFLoader("/content/drive/MyDrive/Colab Notebooks/RAG/DAY1/Demian.pdf")

# PDF 파일을 로딩하여 페이지 단위로 글자를 텍스트 데이터로 분할하여 읽어서 pages 변수에 리스트 형태로 저장.
pages = loader.load_and_split()

# 2. 텍스트 분할 (Text Splitting)----------------------------------------------
# 텍스트를 자르는 기준 설정.
# - chunk_size=500: Chunk의 최대 크기를 500 토큰으로 제한.
# - chunk_overlap=0: Chunk 간에 중복되는 글자가 없도록 설정.
# - length_function=tiktoken_len: 글자 수가 아닌 OpenAI의 토큰 수 계산 방식(tiktoken)을 기준으로 함.
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=0, length_function = tiktoken_len)

# pages를  text_splitter 기준에 맞춰 잘게 쪼개어 docs에 저장
docs = text_splitter.split_documents(pages)


# 3. 벡터 데이터베이스 생성 및 저장 (Vector Store Indexing) -------------------
# 잘게 쪼갠 문서 조각들(docs)을 embedding_model을 사용해 Vector로 변환한 뒤, Chroma Vector DB에 저장.
db = Chroma.from_documents(docs, embedding_model)


# 4. 유사도 검색 (Similarity Search) ------------------------------------------
query = "how Demian look like?"       # 질문 정의

# query를 db에 넣어, 질문의 의미와 '가장 유사한' 문서 조각들을 찾아냄.(상위 결과 몇 개를 리스트 형태)
docs = db.similarity_search(query)

# 검색된 결과 중 가장 유사도가 높은 첫 번째 문서 조각의 실제 텍스트 내용(page_content)을 화면에 출력.
print(docs[0].page_content)


DEMIAN 
with a feeling of nausea, I noticed Demian's expression. 
He had not thrust himself to the front but stood right 
at the back, looking elc!gant and at ease as usual. His 
glance seemed directed at the horse's head, and again it 
. showed that deep, quiet, almost fanatical yet passionate 
absorption. I could not help staring at him for some 
moments and it was then that I felt aware of a very 
uncanny sensation in my remote consciousness. I saw 
Demian's face and remarked that it was not a boy's face 
but a man's and then I saw, or rather became aware, that 
it was not really the face of a man either; it had some­
thing different about it, almost a feminine element. And 
for the time being his face seemed neither masculine 
nor childish, neither old nor young but a hundred years 
old, almost timeless and bearing the mark of other 
periods of history than our own. Animals might look 
thus, trees or stars. I did not know then, of course, I 
did not feel exactly what I am writing a

### Step 4 Retrievers

- 사용자의 질문에 답하는 질의응답 체인(QA Chain)을 생성

In [70]:
from langchain_classic.chains.retrieval_qa.base import RetrievalQA                    # LangChain의 클래식 체인 모듈에서 가장 기본적인 질의응답 시스템
from langchain_core.callbacks.streaming_stdout import StreamingStdOutCallbackHandler  # AI가 답변을 생성할 때 한 글자씩 실시간으로 화면에 출력해 주는 핸들러

# OpenAI 대화형 모델(LLM) 설정
llm = ChatOpenAI(
    model="gpt-4o",              # 메인 모델 지정 : 또는 "gpt-4o-mini"
    temperature=0.0,             # 창의성 조절 (0.0은 가장 일관되고 정확한 답변을 유도)
    streaming=True,              # 실시간 출력 on
    callbacks=[StreamingStdOutCallbackHandler()], # 출력을 콘솔에 바로 뿌려줍니다
)

# 최종 QA 시스템(체인) 구축
qa = RetrievalQA.from_chain_type(llm,                                 # 위에서 정의한 OpenAI 모델을 사용
                                 chain_type="stuff",                  # "stuff" 방식: 검색된 모든 문서 조각을 프롬프트 하나에 통째로 집어넣어 LLM에 전달하는 방식

                                 retriever=db.as_retriever(           # 검색기(Retriever) 설정: DB에서 관련 문서를 어떻게 찾아올지 정의.
                                     search_type="mmr",               # MMR(Maximal Marginal Relevance) 알고리즘 사용
                                     search_kwargs={"k": 3,           # LLM에게 최종적으로 전달할 가장 적합한 문서 조각의 개수 (3개)
                                                    "fetch_k" : 10}), # MMR 알고리즘을 적용하기 위해 DB에서 우선 후보로 뽑아올 문서 개수 (10개)
                                 return_source_documents=True)        # AI의 답변뿐만 아니라, 답변의 근거가 된 '원본 문서 조각들'도 결과에 함께 포함해서 반환하도록 설정


In [71]:
query = "how demian looks like"     # 1. 질문
result = qa(query)                  # 2. 답변 생성

from IPython.display import Markdown, display
display(Markdown(result["result"])) # 3. 최종 답변 출력

Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His appearance is unique and different from others, almost like an animal, a spirit, or an image. There is a suggestion of a feminine element in his features, and his expression is deeply absorbed and passionate. Overall, Demian's appearance is described as being different and enigmatic, making him stand out from those around him.

Demian is described as having a face that is neither distinctly masculine nor childish, neither old nor young, but rather timeless and bearing the mark of other periods of history. His appearance is unique and different from others, almost like an animal, a spirit, or an image. There is a suggestion of a feminine element in his features, and his expression is deeply absorbed and passionate. Overall, Demian's appearance is described as being different and enigmatic, making him stand out from those around him.

### Step 5 Question Answering

In [72]:
query = "Why did Franz Kromer blackmail Sinclair and how did it affect him?"     # 1. 질문
result = qa(query)                  # 2. 답변 생성

from IPython.display import Markdown, display
display(Markdown(result["result"])) # 3. 최종 답변 출력

Franz Kromer blackmailed Sinclair because he had witnessed Sinclair committing a minor theft and used this knowledge to extort money from him. Kromer demanded money from Sinclair, threatening to expose him if he did not comply. This blackmail had a significant impact on Sinclair, causing him fear and anxiety. He felt trapped and powerless under Kromer's control, which led to a sense of isolation and distress. Sinclair was uncomfortable in Kromer's presence and feared his wrath, which further exacerbated his feelings of vulnerability and helplessness.

Franz Kromer blackmailed Sinclair because he had witnessed Sinclair committing a minor theft and used this knowledge to extort money from him. Kromer demanded money from Sinclair, threatening to expose him if he did not comply. This blackmail had a significant impact on Sinclair, causing him fear and anxiety. He felt trapped and powerless under Kromer's control, which led to a sense of isolation and distress. Sinclair was uncomfortable in Kromer's presence and feared his wrath, which further exacerbated his feelings of vulnerability and helplessness.

In [75]:
# 데미안에서 가장 유명한 은유와 상징적 개념들을 대변하는 청크(Chunk)들을 정확히 끄집어내는지 확인
query = "Explain the symbolism of the bird fighting its way out of the egg."     # 1. 질문 : 알을 깨고 나오는 새
result = qa(query)                  # 2. 답변 생성

from IPython.display import Markdown, display
display(Markdown(result["result"])) # 3. 최종 답변 출력

The symbolism of the bird fighting its way out of the egg represents a process of transformation and rebirth. The egg symbolizes the world or a state of being that confines or limits an individual. To be born or to achieve a new state of consciousness, one must break free from these limitations, which involves destruction or overcoming of the old world or self. The bird's struggle signifies the effort and struggle required to achieve this transformation. The bird flying to God, named Abraxas, suggests a journey towards a higher understanding or enlightenment, where Abraxas represents a god that reconciles opposites, such as the godly and the satanic, symbolizing the integration of all aspects of existence.

The symbolism of the bird fighting its way out of the egg represents a process of transformation and rebirth. The egg symbolizes the world or a state of being that confines or limits an individual. To be born or to achieve a new state of consciousness, one must break free from these limitations, which involves destruction or overcoming of the old world or self. The bird's struggle signifies the effort and struggle required to achieve this transformation. The bird flying to God, named Abraxas, suggests a journey towards a higher understanding or enlightenment, where Abraxas represents a god that reconciles opposites, such as the godly and the satanic, symbolizing the integration of all aspects of existence.


----------------------

## titnic.csv 로 검색 테스트
  - 벡터 검색 (MMR, similarity)
  - 키워드 검색 (BM25)
  - 하이브리드 검색 (벡터+키워드)

In [2]:

# CSV loader ----------------
from langchain_community.document_loaders import CSVLoader

loaderCSV = CSVLoader("/content/drive/MyDrive/Colab Notebooks/RAG/DAY1/titanic.csv")
data = loaderCSV.load()


/tmp/ipykernel_95790/2424207275.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import CSVLoader


In [3]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from IPython.display import Markdown

# embedding model 정의
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 타이타닉 승객 데이터를 Chroma 벡터 DB에 저장
db_titanic = Chroma.from_documents(data, embedding_model)

# 답변생성
qa_titanic = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db_titanic.as_retriever(
        search_type="mmr",                      # MMR
        search_kwargs={"k": 3,
                       "fetch_k" : 10
                       }
    ),
    return_source_documents=True
)

image.png

- 4번째에 Futrelle, Mrs. Jacques Heath 가 존재함.

In [19]:
# query = "Tell me about a passenger named 'Braund, Mr. Owen Harris'. Did he survive, what was his cabin class, and how much did he pay for the ticket?"
query = "이름이 'Futrelle, Mrs. Jacques Heath'인 승객의 성별, 객실 등급, 생존 여부를 알려주고, 이 승객이 탑승했을 당시의 하루를 상상해서 짧은 스토리로 들려줘."
result = qa_titanic(query)

display(Markdown(result["result"]))

'Futrelle, Mrs. Jacques Heath'라는 이름의 승객에 대한 정보는 제공된 데이터에 없습니다. 따라서 이 승객의 성별, 객실 등급, 생존 여부에 대한 정보를 알 수 없습니다.

- 분명 Mrs. Jacques Heath Futrelle 가 CSV 4번째에 존재하는데, 못찾고, Mr. Jacques Heath Futrelle만 찾음, 찾았는데 뒤에나오는 남편껄로 엎었나 했다.
- 그런데 알아보니, Mr., Mrs. 는 자주 등장하는 단어로 임베딩시 가중치가 적어 유사도 점수가 거의 동일하고, Mr.가 미세하게 좀더 높았을 가능성이 있으리라 추정.
- Mr., Mrs.처럼  스펠링 하나가 다르면 완전히 다른 의미를 갖게 되는 데이터의 경우, 문맥의 의미를 찾는 게 RAG로는 어려움이 있는 것 같음.

- db_titanic.as_retriever에 **lambda_mult** 파라메터 추가해서 다시 테스트.
  - 질문과의 유사도를 높여서 0.8로 셋팅 후 테스트

In [66]:
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_classic.chains.retrieval_qa.base import RetrievalQA
from IPython.display import Markdown

# embedding model 정의
embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 타이타닉 승객 데이터를 Chroma 벡터 DB에 저장
db_titanic = Chroma.from_documents(data, embedding_model)

# 답변생성
qa_titanic = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db_titanic.as_retriever(
        search_type="mmr",                      # MMR
        search_kwargs={"k": 3,
                       "fetch_k" : 10
                         ,"lambda_mult": 0.8       # <------ 추가(질문과의 유사도를 더 높여 검색)
                       }
    ),
    return_source_documents=True
)

In [54]:
query = "이름이 Futrelle, Mrs. Jacques Heath인 승객의 성별, 객실 등급, 생존 여부를 알려주고, 이 승객이 탑승했을 당시의 하루를 상상해서 짧은 스토리로 들려줘."
result = qa_titanic(query)

display(Markdown(result["result"]))

이름이 Futrelle, Mrs. Jacques Heath인 승객의 성별은 여성이고, 객실 등급은 1등급입니다. 그녀는 생존하지 못했습니다.

이제 그녀가 탑승했을 당시의 하루를 상상해보겠습니다.

---

1912년 4월 14일, 일요일 아침. Futrelle, Mrs. Jacques Heath는 타이타닉의 1등급 객실에서 눈을 떴습니다. 그녀는 창문을 통해 들어오는 부드러운 햇살을 느끼며, 오늘 하루가 또 다른 멋진 항해의 날이 될 것임을 기대했습니다. 그녀는 남편과 함께 아침 식사를 하기 위해 식당으로 향했습니다. 식당에서는 다양한 국적의 승객들이 모여 아침을 즐기고 있었고, 그녀는 그들과 인사를 나누며 즐거운 대화를 나눴습니다.

오전에는 갑판 위를 산책하며 신선한 바닷바람을 만끽했습니다. 그녀는 타이타닉의 웅장함과 아름다움에 감탄하며, 이 여행이 평생 잊지 못할 추억이 될 것이라고 생각했습니다. 점심 후에는 도서관에서 책을 읽으며 여유로운 시간을 보냈습니다.

저녁이 되자, 그녀는 남편과 함께 정찬에 참석했습니다. 우아한 드레스와 보석으로 치장한 그녀는 다른 승객들과 함께 음악을 즐기며, 화려한 만찬을 즐겼습니다. 그날 밤, 그녀는 침대에 누워 내일의 여행을 기대하며 잠이 들었습니다.

하지만 그날 밤, 타이타닉은 빙산과 충돌했고, 그녀의 운명은 영원히 바뀌었습니다.

- **lambda_mult** 추가하고, 값을 0.8로 주니 질문과의 유사도를 높여 검색이 됨.
  - 기본값 0.5인 경우 검색되지 않음.


- 검색방식 변경 (MMR -> similarity)
  - MMR(Maximal Marginal Relevance)
    - 질문과 관련성도 높고, 이미 선택된 문서들과 너무 중복되지 않는 문서를 고르는 방식으로 다양한 결과가 나올 수 있음.
    - 비슷한 것만 반복해서 가져오는 문제를 줄이는 검색 방식.
      1. 질문과 얼마나 비슷한가?
      2. 이미 뽑은 문서들과 얼마나 다른가?
  - Similarity
    - 질문 벡터와 문서 벡터의 유사도를 계산해서 가장 가까운 문서 TOP-K.

In [6]:
# similarity 유사도 검색으로 변경
qa_titanic = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=db_titanic.as_retriever(
        search_type="similarity", # mmr 대신 '단순 유사도 검색'으로 변경!
        search_kwargs={"k": 5}    # 비슷한 것 5개 가져오기.
    ),
    return_source_documents=True
)

In [7]:
query = "이름이 Futrelle, Mrs. Jacques Heath인 승객의 성별, 객실 등급, 생존 여부를 알려주고, 이 승객이 탑승했을 당시의 하루를 상상해서 짧은 스토리로 들려줘."
result = qa_titanic(query)

display(Markdown(result["result"]))

이름이 Futrelle, Mrs. Jacques Heath인 승객의 성별은 여성이고, 객실 등급은 1등급(Pclass: 1)이며, 생존했습니다.

스토리:
1912년 4월 14일, Futrelle, Mrs. Jacques Heath는 타이타닉 호의 호화로운 1등급 객실에서 하루를 시작했습니다. 그녀는 남편과 함께 아침 식사를 즐기며 대서양을 가로지르는 여행의 낭만을 만끽했습니다. 그날 오후, 그녀는 다른 승객들과 함께 갑판 위를 산책하며 신선한 바닷바람을 즐겼습니다. 저녁에는 우아한 드레스를 입고 1등급 식당에서 정찬을 즐기며, 새로운 친구들과 대화를 나누었습니다. 그날 밤, 그녀는 침대에 들기 전 창밖으로 별이 빛나는 하늘을 바라보며 이 여행이 가져다줄 새로운 모험을 기대했습니다. 그러나 그녀는 곧 다가올 사건을 전혀 예상하지 못했습니다. 다행히도, 그녀는 그날 밤의 비극적인 사건에서 살아남아 이후의 삶을 이어갈 수 있었습니다.

- MMR -> similarity로  검색방식 변경하니 잘 찾음.
- 데이터의 성격에 따라 검색 방식을 다르게 해야함을 알았다.


- 하이브리드 검색
  - BM25
    - 문서 검색에서 많이 쓰는 **키워드 기반 검색** 알고리즘
    - 사용자가 입력한 **질문에 들어 있는 단어가 문서안에 잘 들어 있는지** 점수 매겨 점수가 높은 문서를 찾아 주는 방식.
    - 의미는 비슷하지만 단어가 다르면 잘 못 찾음.
  - Chroma 벡터검색
    - 의미 검색


In [18]:
!pip install rank_bm25
!pip install -U langchain

In [12]:
# !pip install -qU langchain langchain-community langchain-core langchain-openai rank_bm25
!pip install -U langchain langchain-community langchain-openai langchain-classic rank_bm25

In [7]:
# 1. 하이브리드 검색을 위한 라이브러리
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever
from langchain_openai import OpenAIEmbeddings
from langchain_classic.chains import RetrievalQA

embedding_model = OpenAIEmbeddings(model="text-embedding-3-small")

# 2. 키워드 검색기 (BM25) 설정
bm25_retriever = BM25Retriever.from_documents(data)   # titanic.csv
bm25_retriever.k = 3                                  # 키워드 검색으로 상위 3개 찾기

# 3. 벡터 DB (Chroma) 설정
# db_titanic = Chroma.from_documents(data, embedding_model)
vector_retriever = db_titanic.as_retriever(search_kwargs={"k": 3})

# 4. 하이브리드 검색기(Ensemble) 결합!
# weights를 통해 반영 비율을 조절가능 (키워드 결과 50%, 벡터 결과 50%)
ensemble_retriever = EnsembleRetriever(
    retrievers=[bm25_retriever, vector_retriever],
    weights=[0.5, 0.5]
)

# 5. QA 체인에 하이브리드 검색기 장착
qa_titanic_hybrid = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=ensemble_retriever, # 위 앙상블 검색기 설정.
    return_source_documents=True
)

In [12]:
query = "이름이 Futrelle, Mrs. Jacques Heath인 승객의 성별, 객실 등급, 생존 여부를 알려주고, 이 승객이 탑승했을 당시의 하루를 상상해서 짧은 스토리로 들려줘."
result = qa_titanic_hybrid(query)

display(Markdown(result["result"]))

이름이 Futrelle, Mrs. Jacques Heath인 승객의 성별은 여성이고, 객실 등급은 1등급이며, 생존하였습니다.

스토리:
1912년 4월의 어느 날, Lily May Peel로도 알려진 Futrelle, Mrs. Jacques Heath는 타이타닉 호의 1등급 객실에서 아침을 맞이했습니다. 그녀는 남편과 함께 여행 중이었고, 그날 아침은 햇살이 창문을 통해 부드럽게 들어오는 평화로운 날이었습니다. Lily는 고급스러운 식당에서 아침 식사를 즐기며, 다른 승객들과 담소를 나누었습니다. 그녀는 타이타닉의 웅장함과 편안함에 감탄하며, 남편과 함께 갑판을 산책하며 바다의 광활함을 만끽했습니다. 저녁이 되자, 그녀는 남편과 함께 저녁 식사를 하고, 그날 밤의 공연을 기대하며 객실로 돌아갔습니다. 그러나 그날 밤, 타이타닉은 빙산과 충돌했고, Lily는 침착하게 구조 보트에 탑승하여 생존할 수 있었습니다. 그녀는 남편과 헤어져야 했지만, 그날의 기억은 그녀의 마음속에 영원히 남아 있었습니다.

- 정확하게 잘 찾음.
- 의미보다는 정확한 이름 매칭이라 BM25 영향으로 검색에 성공한듯.

In [13]:
query = "1등석 여성 생존자는 누구야?"
result = qa_titanic_hybrid(query)

display(Markdown(result["result"]))

1등석 여성 생존자는 다음과 같습니다:

1. Bonnell, Miss. Elizabeth
2. Leader, Dr. Alice (Farnham)
3. Bowerman, Miss. Elsie Edith

- 벡터로 의미파악하여 검색.
- BM25 보다는 벡터의 영향이 컸을 듯.

In [16]:
query = "1등석 여성 생존자는 누구야?"

print("========BM25 결과========")
for doc in bm25_retriever.invoke(query):
    print(doc.page_content)

print("\n========VECTOR 결과========")
for doc in vector_retriever.invoke(query):
    print(doc.page_content)

========BM25 결과========
PassengerId: 1
Survived: 0
Pclass: 3
Name: Braund, Mr. Owen Harris
Sex: male
Age: 22
SibSp: 1
Parch: 0
Ticket: A/5 21171
Fare: 7.25
Cabin: 
Embarked: S
PassengerId: 891
Survived: 0
Pclass: 3
Name: Dooley, Mr. Patrick
Sex: male
Age: 32
SibSp: 0
Parch: 0
Ticket: 370376
Fare: 7.75
Cabin: 
Embarked: Q
PassengerId: 890
Survived: 1
Pclass: 1
Name: Behr, Mr. Karl Howell
Sex: male
Age: 26
SibSp: 0
Parch: 0
Ticket: 111369
Fare: 30
Cabin: C148
Embarked: C

========VECTOR 결과========
PassengerId: 12
Survived: 1
Pclass: 1
Name: Bonnell, Miss. Elizabeth
Sex: female
Age: 58
SibSp: 0
Parch: 0
Ticket: 113783
Fare: 26.55
Cabin: C103
Embarked: S
PassengerId: 797
Survived: 1
Pclass: 1
Name: Leader, Dr. Alice (Farnham)
Sex: female
Age: 49
SibSp: 0
Parch: 0
Ticket: 17465
Fare: 25.9292
Cabin: D17
Embarked: S
PassengerId: 357
Survived: 1
Pclass: 1
Name: Bowerman, Miss. Elsie Edith
Sex: female
Age: 22
SibSp: 0
Parch: 1
Ticket: 113505
Fare: 55
Cabin: E33
Embarked: S


- 결과 정리

| 검색 방식  | 나온 승객                       | Survived | Pclass | Sex    | 조건 만족 여부 |
| ------ | --------------------------- | -------: | -----: | ------ | -------- |
| BM25   | Braund, Mr. Owen Harris     |        0 |      3 | male   | X 불일치    |
| BM25   | Dooley, Mr. Patrick         |        0 |      3 | male   | X  불일치    |
| BM25   | Behr, Mr. Karl Howell       |        1 |      1 | male   | X 일부만 일치 |
| Vector | Bonnell, Miss. Elizabeth    |        1 |      1 | female | O 일치     |
| Vector | Leader, Dr. Alice (Farnham) |        1 |      1 | female | O 일치     |
| Vector | Bowerman, Miss. Elsie Edith |        1 |      1 | female | O 일치     |


- 벡터가 검색에 영향을 줌.


In [14]:
query = "Pclass 1 Sex female Survived 1"
result = qa_titanic_hybrid(query)

display(Markdown(result["result"]))

The data provided includes several female passengers in Pclass 1 who survived. Here are some examples:

1. PassengerId: 258
   - Name: Cherry, Miss. Gladys
   - Age: 30
   - Ticket: 110152
   - Fare: 86.5
   - Cabin: B77
   - Embarked: S

2. PassengerId: 836
   - Name: Compton, Miss. Sara Rebecca
   - Age: 39
   - Ticket: PC 17756
   - Fare: 83.1583
   - Cabin: E49
   - Embarked: C

3. PassengerId: 559
   - Name: Taussig, Mrs. Emil (Tillie Mandelbaum)
   - Age: 39
   - Ticket: 110413
   - Fare: 79.65
   - Cabin: E67
   - Embarked: S

These passengers were all in first class, female, and survived.

- 검색에 BM25의 영향도가 컸을 것으로 예상.
- 벡터로는 Pclass 1 Sex female Survived 1

In [17]:
query = "Pclass 1 Sex female Survived 1"

print("========BM25 결과========")
for doc in bm25_retriever.invoke(query):
    print(doc.page_content)

print("\n========VECTOR 결과========")
for doc in vector_retriever.invoke(query):
    print(doc.page_content)

========BM25 결과========
PassengerId: 173
Survived: 1
Pclass: 3
Name: Johnson, Miss. Eleanor Ileen
Sex: female
Age: 1
SibSp: 1
Parch: 1
Ticket: 347742
Fare: 11.1333
Cabin: 
Embarked: S
PassengerId: 836
Survived: 1
Pclass: 1
Name: Compton, Miss. Sara Rebecca
Sex: female
Age: 39
SibSp: 1
Parch: 1
Ticket: PC 17756
Fare: 83.1583
Cabin: E49
Embarked: C
PassengerId: 559
Survived: 1
Pclass: 1
Name: Taussig, Mrs. Emil (Tillie Mandelbaum)
Sex: female
Age: 39
SibSp: 1
Parch: 1
Ticket: 110413
Fare: 79.65
Cabin: E67
Embarked: S

========VECTOR 결과========
PassengerId: 258
Survived: 1
Pclass: 1
Name: Cherry, Miss. Gladys
Sex: female
Age: 30
SibSp: 0
Parch: 0
Ticket: 110152
Fare: 86.5
Cabin: B77
Embarked: S
PassengerId: 531
Survived: 1
Pclass: 2
Name: Quick, Miss. Phyllis May
Sex: female
Age: 2
SibSp: 1
Parch: 1
Ticket: 26360
Fare: 26
Cabin: 
Embarked: S
PassengerId: 581
Survived: 1
Pclass: 2
Name: Christy, Miss. Julie Rachel
Sex: female
Age: 25
SibSp: 1
Parch: 1
Ticket: 237789
Fare: 30
Cabin: 
Embark


- 검색 결과

| 검색 방식  | PassengerId | 이름                                     | Survived | Pclass | Sex    | 조건 만족 여부     |
| ------ | ----------: | -------------------------------------- | -------: | -----: | ------ | ------------ |
| BM25   |         173 | Johnson, Miss. Eleanor Ileen           |        1 |      3 | female | X Pclass 불일치 |
| BM25   |         836 | Compton, Miss. Sara Rebecca            |        1 |      1 | female | O 일치         |
| BM25   |         559 | Taussig, Mrs. Emil (Tillie Mandelbaum) |        1 |      1 | female | O 일치         |
| Vector |         258 | Cherry, Miss. Gladys                   |        1 |      1 | female | O 일치         |
| Vector |         531 | Quick, Miss. Phyllis May               |        1 |      2 | female | X Pclass 불일치 |
| Vector |         581 | Christy, Miss. Julie Rachel            |        1 |      2 | female | X Pclass 불일치 |

- BM25는 3개중 2개, 벡터는 3개중 1개가 맞음.
- BM25가 위 자연어로 질문했을 때보다 좀더 잘 작동함.
- 아마도 질문에 Pclass, Sex, female 단어가 들어가서 인듯함.